# University of London - ML Code - Computer Science Final Project

**BSc Computer Science**

**Subject: CM3070 Computer Science Final Project**

**Student: In Final Project report**

**Student Number: In Final Project report**

## Genetic Algorihtm

The implementation of the Genetic Algorithm based on the best model structure obtained in the ml_process notebook.

For the first time, all currency pairs available are used and not only the USD-JPY one.

The result of this notebook are 3 models per currency pair (each with a delay longer than the previous one).

## Preparing the data

### Currencies

In [ ]:
currencies = [
    "jpy",
    "aud",
    "cad",
    "cny",
    "hkd",
    "chf",
]

### Transforming the csv data to a numpy array

In [ ]:
import numpy as np

str_to_np_date = lambda x: np.datetime64(x)

currencies_raw_data = {}

# Pairs of currencies
for currency in currencies:
    raw_data = np.genfromtxt(f"./data/currency-data/USD-{currency.upper()}-DAILY.csv", skip_header=1, delimiter=";", usecols=1)
    raw_data_dates = np.genfromtxt(f"./data/currency-data/USD-{currency.upper()}-DAILY.csv", skip_header=1, delimiter=";", usecols=0, converters={0: str_to_np_date})
    
    # As the currency data is from newer to older, the order should be inverted.
    raw_data = np.flip(raw_data, axis=0)
    raw_data_dates = np.flip(raw_data_dates, axis=0)

    # calculate train sample number
    train_samples_number = len(raw_data)

    currencies_raw_data[f"{currency}"] = {
        f"usd_{currency}_train_samples_number": train_samples_number,
        f"usd_{currency}_raw_data": raw_data,
        f"usd_{currency}_raw_data_dates": raw_data_dates
    }

    # Print essential data information
    print(f"Length usd_{currency}_raw_data: ",len(raw_data))
    print(f"Data type usd_{currency}_raw_data: ",raw_data.dtype)
    print(f"Raw Data Sample - usd_{currency}_raw_data: ",raw_data[:10])
    print(f"Raw Data Dates Sample - usd_{currency}_raw_data_dates: ",raw_data_dates[:10])
    print(f"Number of train samples - usd_{currency}_raw_data_dates: ", train_samples_number)

# Check one of the currencies in the dictionary
print(currencies_raw_data["jpy"][f"usd_jpy_raw_data"][:10])
print(currencies_raw_data["jpy"][f"usd_jpy_raw_data_dates"][:10])

print(currencies_raw_data)

### Creating timeseries data

Based on the code provided in Chapter 10 (Deep Learning For Timeseries) of the book Deep Learning with Python by Francois Chollet [1].

In [ ]:
from tensorflow import keras
import tensorflow as tf

# Parameters: sampling_rate, sequence_length, delay, and batch_size
sampling_rate = 1
sequence_length = 150 # Observations will go back 150 days
delays = []

for i in range(3):
    
    delays.append(sampling_rate * (sequence_length + (5 + (20 * i)) - 1)) # target is 5, 25, 45 days after the end of the sequence

train_datasets = {}

data_inputs_test_all = {}
data_outputs_test_all = {}
data_inputs_all = {}
data_outputs_all = {}

for index, delay in enumerate(delays):
    for currency in currencies:
        # train dataset
        train_dataset = keras.utils.timeseries_dataset_from_array(
            currencies_raw_data[currency][f"usd_{currency}_raw_data"],
            targets=currencies_raw_data[currency][f"usd_{currency}_raw_data"][delay:],
            sampling_rate=sampling_rate,
            sequence_length=sequence_length,
            batch_size=currencies_raw_data[currency][f"usd_{currency}_train_samples_number"],
        )

        if index == 0:
            train_datasets[f"train_dataset_usd_{currency}_delay_5"] = train_dataset
        if index == 1:
            train_datasets[f"train_dataset_usd_{currency}_delay_25"] = train_dataset
        if index == 2:
            train_datasets[f"train_dataset_usd_{currency}_delay_45"] = train_dataset

        # Extracting data inputs and outputs
        # Based on the code provided in Chapter 10 (Deep Learning For Timeseries) of the book Deep Learning with Python by Francois Chollet [1].
        
        for samples, targets in train_dataset:
            # print("Samples: ", samples)
            # print("Sample shape: ", samples.shape)
            # print("Targets: ", targets)
            # print("Targets shape: ", targets.shape)
            data_inputs = tf.make_ndarray(tf.make_tensor_proto(samples))
            data_outputs = tf.reshape(tf.make_ndarray(tf.make_tensor_proto(targets)), [-1,1])

            if index == 0:
                data_inputs_test_all[f"{currency}_delay_5"] = data_inputs[-200:]
                data_outputs_test_all[f"{currency}_delay_5"] = data_outputs[-200:]
                data_inputs_all[f"{currency}_delay_5"] = data_inputs[:-200]
                data_outputs_all[f"{currency}_delay_5"] = data_outputs[:-200]
            if index == 1:
                data_inputs_test_all[f"{currency}_delay_25"] = data_inputs[-200:]
                data_outputs_test_all[f"{currency}_delay_25"] = data_outputs[-200:]
                data_inputs_all[f"{currency}_delay_25"] = data_inputs[:-200]
                data_outputs_all[f"{currency}_delay_25"] = data_outputs[:-200]
            if index == 2:
                data_inputs_test_all[f"{currency}_delay_45"] = data_inputs[-200:]
                data_outputs_test_all[f"{currency}_delay_45"] = data_outputs[-200:]
                data_inputs_all[f"{currency}_delay_45"] = data_inputs[:-200]
                data_outputs_all[f"{currency}_delay_45"] = data_outputs[:-200]

### - Checking that timeseries data works correctly

Based on the code provided in Chapter 10 (Deep Learning For Timeseries) of the book Deep Learning with Python by Francois Chollet [1].

In [ ]:
for inputs, targets in train_datasets["train_dataset_usd_jpy_delay_5"]:
    for i in range(inputs.shape[0])[:10]:
        print([float(x) for x in inputs[i]], float(targets[i]))

Check data inputs and outputs

In [ ]:
for index, delay in enumerate(delays):
    real_delay = 0
    if index == 0:
        real_delay = 5
    if index == 1:
        real_delay = 25
    if index == 2:
        real_delay = 45
    print(f"---- ---- ---- ---- ---- ---- ---- ----\nDelay: {real_delay} days.")
    for currency in currencies:
        print(f"---- ---- ---- ----\nCurrency: {currency}.")
        print("Data Inputs: ", len(data_inputs_all[f"{currency}_delay_{real_delay}"]))
        print("Data Inputs Test: ", len(data_inputs_test_all[f"{currency}_delay_{real_delay}"]))
        print("Data Outputs: ", len(data_outputs_all[f"{currency}_delay_{real_delay}"]))
        print("Data Outputs Test: ", len(data_outputs_test_all[f"{currency}_delay_{real_delay}"]))
    print(f"---- ---- ---- ---- ---- ---- ---- ----")

## Genetic Algorithm

The code for the Genetic Algorithm is based on the code provided by the PyGAD library documentation [5].

### - LSTM Model (based on best model obtained from running the tuner)

Get structure of best model

In [ ]:
from keras.models import load_model

loaded_model = load_model("models/ml_process_tuner_best_model.keras")

loaded_model.summary()

# Get which activation functions were used
for layer in loaded_model.layers:
    print(layer.activation)

Create a new model based on the structure of the best model

In [ ]:
from keras import models
from keras import layers
from keras import activations

def build_lstm_model():
    model = models.Sequential()
    model.add(layers.LSTM(64, input_shape=(sequence_length, 1)))
    model.add(layers.Dense(32, activation=activations.relu))
    model.add(layers.Dense(28, activation=activations.relu))
    model.add(layers.Dense(1, activation=activations.linear))
    model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
    model.summary()
    return model

lstm_model = build_lstm_model()

### - Genetic Algorithm implementation

In [ ]:
import pygad.kerasga

# - Instance of the pygad.kerasga.KerasGA class
keras_ga = pygad.kerasga.KerasGA(model=lstm_model, num_solutions=10)

# Genetic Algorithm
for index, delay in enumerate(delays):
    real_delay = 0
    if index == 0:
        real_delay = 5
    if index == 1:
        real_delay = 25
    if index == 2:
        real_delay = 45
    print(f"---- ---- ---- ---- ---- ---- ---- ----\nDelay: {real_delay} days.")
    for currency in currencies:
        print(f"---- ---- ---- ----\nCurrency: {currency}.")

        # - Fitness function
        def fitness_function(ga_instance, solution, solution_index):
            global keras_ga, lstm_model
            predictions = pygad.kerasga.predict(model=lstm_model, solution=solution, data=data_inputs_all[f"{currency}_delay_{real_delay}"])
            mae = keras.losses.MeanAbsoluteError()
            absolute_error = mae(data_outputs_all[f"{currency}_delay_{real_delay}"], predictions).numpy() + 0.00000001
            solution_fitness = 20 / absolute_error
            return solution_fitness

        # - Track GA
        def on_generation(ga_instance):
            print(f"Generation = {ga_instance.generations_completed}")
            print(f"Fitness    = {ga_instance.best_solution()[1]}")

        # - Create instance of the pygad.GA class
        num_generations = 100 # reduced from 400 to 100 as there was already the tuner stage selecting the best model structure, and in order to reduce time as many models need to be created
        num_parents_mating = 4
        initial_population = keras_ga.population_weights

        ga_instance = pygad.GA(
                                num_generations=num_generations,
                                num_parents_mating=num_parents_mating,
                                initial_population=initial_population,
                                fitness_func=fitness_function,
                                on_generation=on_generation,
                                suppress_warnings=True,
                                parallel_processing=["thread", 4],
                                mutation_type="adaptive",
                                mutation_percent_genes=[8,4]
                            )

        # - Run
        ga_instance.summary()

        ga_instance.run()

        print(ga_instance.plot_fitness(title=f"Genetic Agorithm - Fitness - {currency}_delay_{real_delay}", linewidth=2))

        # - Run best solution
        solution, solution_fitness, solution_index = ga_instance.best_solution()
        predictions = pygad.kerasga.predict(model=lstm_model, solution=solution,data=data_inputs_all[f"{currency}_delay_{real_delay}"])
        mae = keras.losses.MeanAbsoluteError()
        absolute_error = mae(data_outputs_all[f"{currency}_delay_{real_delay}"], predictions).numpy()
        print("Solution fitness: ", solution_fitness, " (20 is the best)")
        print("Solution Predictions: ", predictions)
        print("Solution Absolute Error: ", absolute_error)

        # - Save Genetic Algorithm
        ga_instance.save(filename=f"genetic-algorithms/genetic_algorithm_{currency}_delay_{real_delay}")

        # ---- ---- Export model (keras and tensorflow) ---- ----
        # Get the best solution as a keras weight matrix
        ga_best_solution_weights = pygad.kerasga.model_weights_as_matrix(model=lstm_model, weights_vector=solution)

        # Set the weights
        lstm_model.set_weights(ga_best_solution_weights)

        # Save the model (for Keras)
        lstm_model.save(f"models/model_{currency}_delay_{real_delay}.keras")

        # Save the model (for Tensorflow)
        lstm_model.export(f"models/model_{currency}_delay_{real_delay}_tf", format="tf_saved_model")

        # ---- ---- ---- ----

## About the code

The Genetic Algorithm code is based on the code shown in the docs of the PyGAD library.

The timeseries data code is based on the code shown in the chapter of the book Deep Learning with Python.

## References

1- Francois Chollet. 2021. Deep Learning with Python, Second Edition. Chapter 10, Deep learning for timeseries, Preparing the data. Retrieved from https://learning.oreilly.com/library/view/deep-learning-with/9781617296864/Text/10.htm#heading_id_5

2- Francois Chollet. 2021. Deep Learning with Python, Second Edition. Chapter 10, Deep learning for timeseries, Let’s try a basic machine learning model. Retrieved from https://learning.oreilly.com/library/view/deep-learning-with/9781617296864/Text/10.htm#heading_id_7

3- Francois Chollet. 2021. Deep Learning with Python, Second Edition. Chapter 10, Deep learning for timeseries, Let’s try a 1D convolutional model. Retrieved from https://learning.oreilly.com/library/view/deep-learning-with/9781617296864/Text/10.htm#heading_id_8

4- Keras. Getting started with KerasTuner. Retrieved from https://keras.io/keras_tuner/getting_started/

5- PyGAD. pygad.kerasga Module. Retrieved from https://pygad.readthedocs.io/en/latest/kerasga.html#